# Run BuffetIQ in Colab for the showcase

This notebook starts the whole BuffetIQ prototype in Google Colab and gives a public link for each page, so the kitchen board can run on the TV, the runner page on a phone, and assessors can scan the guest page QR code. Run the cells in order. It takes about 5 minutes the first time.

**Before you start:** a share link lasts 72 hours and anyone who has it can open the app, so launch it on the day of the showcase and never enter real guest data. Everything in BuffetIQ is fictional.

## 1. Get the code and install it

A GPU is not needed to run the app. Choose **Runtime → Change runtime type → T4 GPU** only if you will also train the camera model here.

In [ ]:
import os, sys
if not os.path.exists('/content/BuffetIQ'):
    !git clone -q -b claude/zen-keller-p86mcy https://github.com/stamedytweet-bit/BuffetIQ.git /content/BuffetIQ
%cd /content/BuffetIQ
!pip install -q -r requirements.txt
sys.path.insert(0, '/content/BuffetIQ')
sys.path.insert(0, '/content/BuffetIQ/src')
print('Installed.')

## 2. Add the Gemini key (optional)

In the left bar, click the key icon (**Secrets**) and add two secrets with notebook access switched on: `GEMINI_API_KEY` (from aistudio.google.com) and `GEMINI_MODEL` (the current free-tier Flash or Flash-Lite model ID). Without them, Ask BuffetIQ still works with basic answers.

In [ ]:
try:
    from google.colab import userdata
    for name in ('GEMINI_API_KEY', 'GEMINI_MODEL'):
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass
except ImportError:
    pass
print('Gemini answers' if os.environ.get('GEMINI_API_KEY') and os.environ.get('GEMINI_MODEL') else 'Basic answers (no Gemini key found)')

## 3. The camera model

If you trained `tray_cls.pt` on the laptop, upload it in the next cell when asked (or press Cancel). Otherwise this cell trains the stand-in model on computer-drawn trays, so the camera page works; its results are labelled as coming from a test model.

In [ ]:
from pathlib import Path
if not Path('models/tray_cls.pt').exists():
    try:
        from google.colab import files
        print('Upload models/tray_cls.pt if you have it, or press Cancel.')
        up = files.upload()
        for name, data in up.items():
            Path('models').mkdir(exist_ok=True)
            Path('models/tray_cls.pt').write_bytes(data)
    except Exception as e:
        print('No upload:', e)
if not Path('models/tray_cls.pt').exists():
    !cd src && python -m buffetiq.camera standin
print('Camera model:', 'trained on photos' if Path('models/tray_cls.pt').exists() else 'stand-in')

## 4. Start the app with the showcase morning

This prints a link for each page. Open the kitchen board on the laptop connected to the TV and press F11 for full screen, open the About page in a second window for the presenter controls, and open the runner page on your phone. Keep this browser tab open: closing Colab stops the app.

In [ ]:
import app
info = app.launch(share=True, showcase_morning=True, block=False)

## 5. QR codes for phones

The guest QR code is for assessors; the runner QR code is for your phone. Through the share link the phone microphone works, because the link is secure (https).

In [ ]:
from IPython.display import display, HTML
from buffetiq import qr
import base64, io
base = (info['share'] or info['local']).rstrip('/')
for label, path in (('Guest page', '/guest'), ('Runner page', '/runner')):
    img = qr.qr_image(base + path, box=6)
    buf = io.BytesIO(); img.save(buf, format='PNG')
    display(HTML(f"<h3>{label}</h3><p>{base + path}</p><img src='data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}' width='220'>"))
print('Printable key-card sleeve:', qr.sleeve(base + '/guest'))

## 6. If something goes wrong

* The links stop working: run cell 4 again (it starts a fresh share link and a fresh showcase morning).
* The board shows the wrong time: use **Reset to 06:15** on the About page.
* Wi-Fi or Colab fails on the day: play the backup video (see the README checklist).